In [1]:
import sys
import os
import pandas as pd
import random
import copy
import pickle
import math
import numpy as np
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import torch
import pykeops
from pykeops.torch import LazyTensor
import torch.distributions as dist

# import cudf
# from cuml.cluster import DBSCAN, HDBSCAN

from datasets.metadata import uci_datasets_info

%matplotlib inline
torch.set_printoptions(precision=3)

/home/bdezoysa/.conda/envs/py310/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
def keops_knn_geo(dataset: torch.Tensor, k: int = 10, device: str = 'cuda'):
    N, D = dataset.shape

    x_i = LazyTensor(dataset.view(N, 1, D))
    y_j = LazyTensor(dataset.view(1, N, D))

    D_ij = (x_i - y_j).abs().sum(dim=-1)
    idx = D_ij.argKmin(K=k, dim=1)

    return idx

def keops_knn_pal_foe(histogram: torch.Tensor, position: torch.Tensor, k: int = 10, device: str = 'cuda'):
    N, hD = histogram.shape
    N, pD = position.shape
    
    density = histogram / histogram.sum(dim=1, keepdim=True)

    xp_i = LazyTensor(position.view(N, 1, pD))
    yp_j = LazyTensor(position.view(1, N, pD))

    MAN = (xp_i - yp_j).abs().sum(dim=-1) / pD

    P = LazyTensor(density.view(N, 1, hD))
    Q = LazyTensor(density.view(1, N, hD))

    M = 0.5 * (P + Q)
    
    eps = 1e-12
    JSD = (0.5 * ( (P * (P + eps).log() - P * (M + eps).log()).sum(dim=-1) + (Q * (Q + eps).log() - Q * (M + eps).log()).sum(dim=-1) )).sqrt()

    pal_D_ij = (1.0 - MAN) + JSD
    foe_D_ij = MAN + (torch.sqrt(torch.log(torch.tensor(2, device=device))) - JSD)

    pal_idx = pal_D_ij.argKmin(K=k, dim=1)
    foe_idx = foe_D_ij.argKmin(K=k, dim=1)

    return pal_idx, foe_idx

def keops_knn_pal(histogram: torch.Tensor, position: torch.Tensor, k: int = 10, device: str = 'cuda'):
    N, hD = histogram.shape
    N, pD = position.shape
    
    density = histogram / histogram.sum(dim=1, keepdim=True)

    xp_i = LazyTensor(position.view(N, 1, pD))
    yp_j = LazyTensor(position.view(1, N, pD))

    MAN = (xp_i - yp_j).abs().sum(dim=-1) / pD

    P = LazyTensor(density.view(N, 1, hD))
    Q = LazyTensor(density.view(1, N, hD))

    M = 0.5 * (P + Q)
    
    eps = 1e-12
    JSD = (0.5 * ( (P * (P + eps).log() - P * (M + eps).log()).sum(dim=-1) + (Q * (Q + eps).log() - Q * (M + eps).log()).sum(dim=-1) )).sqrt()

    pal_D_ij = (1.0 - MAN) + JSD

    pal_idx = pal_D_ij.argKmin(K=k, dim=1)

    return pal_idx

In [3]:
def KMeans(x, K=10, Niter=10, tol=1e-6, use_cuda=True, verbose=True):
    """Implements Lloyd's algorithm for the Euclidean metric."""

    start = time.time()
    N, D = x.shape  # Number of samples, dimension of the ambient space

    # c = x[:K, :].clone()  # Simplistic initialization for the centroids
    c_indices = torch.randperm(len(x))[:K] # Get K random unique indices
    c = x[c_indices].clone()

    x_i = LazyTensor(x.view(N, 1, D))  # (N, 1, D) samples
    c_j = LazyTensor(c.view(1, K, D))  # (1, K, D) centroids

    # K-means loop:
    # - x  is the (N, D) point cloud,
    # - cl is the (N,) vector of class labels
    # - c  is the (K, D) cloud of cluster centroids
    for i in range(Niter):
        c_old = c.clone()

        # E step: assign points to the closest cluster -------------------------
        D_ij = ((x_i - c_j) ** 2).sum(-1)  # (N, K) symbolic squared distances
        cl = D_ij.argmin(dim=1).long().view(-1)  # Points -> Nearest cluster

        # M step: update the centroids to the normalized cluster average: ------
        # Compute the sum of points per cluster:
        c.zero_()
        c.scatter_add_(0, cl[:, None].repeat(1, D), x)

        # Divide by the number of points per cluster:
        Ncl = torch.bincount(cl, minlength=K).type_as(c).view(K, 1)
        c /= Ncl  # in-place division to compute the average

        movement = (c - c_old).norm()
        if movement < tol:
            print(f"Converged at iteration {i}")
            break

    if verbose:  # Fancy display -----------------------------------------------
        if use_cuda:
            torch.cuda.synchronize()
        end = time.time()
        print(
            f"K-means for the Manhattan metric with {N:,} points in dimension {D:,}, K = {K:,}:"
        )
        print(
            "Timing for {} iterations: {:.5f}s = {} x {:.5f}s\n".format(
                Niter, end - start, Niter, (end - start) / Niter
            )
        )

    return cl, c, end - start

In [4]:
def preprocess_dataframe(df, target_col, cat_cols, id_cols=None):
    # Create a copy to avoid modifying the original dataframe in-place
    df = df.copy()
    
    # 0. Drop ID Columns
    if id_cols:
        # Only drop columns that actually exist in the dataframe to prevent errors
        cols_to_drop = [col for col in id_cols if col in df.columns]
        df = df.drop(columns=cols_to_drop)
        
    # 1. Drop rows with missing values (NaNs)
    # We do this after dropping IDs so we don't accidentally drop a good row 
    # just because it was missing an ID we didn't need anyway.
    df = df.dropna().reset_index(drop=True)
    
    # 2. Encode Non-Numeric Target Variable
    target_label_mapping = {}
    if target_col in cat_cols and not pd.api.types.is_numeric_dtype(df[target_col]):
        unique_classes = df[target_col].unique()
        target_label_mapping = {val: i for i, val in enumerate(unique_classes)}
        df[target_col] = df[target_col].map(target_label_mapping)
    
    # 3. Create Mapping & Rename
    features = [col for col in df.columns if col != target_col]
    
    col_name_mapping = {target_col: 'x0'}
    for i, col in enumerate(features, start=1):
        col_name_mapping[col] = f'x{i}'
        
    df = df.rename(columns=col_name_mapping)
    
    # 4. Order columns: target variable ('x0') first
    ordered_cols = ['x0'] + [f'x{i}' for i in range(1, len(features) + 1)]
    df = df[ordered_cols]
    
    # 5. One-Hot Encoding
    # Get the new names for the categorical columns, excluding the target
    mapped_cat_cols = [col_name_mapping[col] for col in cat_cols if col in col_name_mapping and col != target_col]
    
    if mapped_cat_cols:
        df = pd.get_dummies(df, columns=mapped_cat_cols, dtype=float)
        
    # 6. Min-Max Normalization
    cols_to_normalize = list(df.columns)
    
    # Exclude the target variable from normalization IF it was flagged as categorical
    if target_col in cat_cols and 'x0' in cols_to_normalize:
        cols_to_normalize.remove('x0')
        
    # Apply Min-Max scaling: (x - min) / (max - min)
    for col in cols_to_normalize:
        col_min = df[col].min()
        col_max = df[col].max()

        # print(col_min, col_max)
        
        if col_max != col_min:
            df[col] = (df[col] - col_min) / (col_max - col_min)
        else:
            df[col] = 0.0 
            
    return df, col_name_mapping, target_label_mapping

In [6]:
torch.manual_seed(20260415)

# num_bins = 64
for name in ['insurance']:#uci_datasets_info:
    pocket_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
    # print(name)
    # if uci_datasets_info[name]['size'] <= 10000:
    #     pocket_sizes = [4, 8, 16, 32]
    # elif uci_datasets_info[name]['size'] <= 50000:
    #     pocket_sizes = [8, 16, 32, 64]
    # elif uci_datasets_info[name]['size'] <= 100000:
    #     pocket_sizes = [16, 32, 64, 128]
    # elif uci_datasets_info[name]['size'] <= 250000:
    #     pocket_sizes = [256, 512, 1024, 2048]

    try:
        df = pd.read_csv(f'./datasets/{name}.csv', delimiter=uci_datasets_info[name]['delim'])
        # print(df.shape)
        # print(df.columns)
        df, col_mapping, target_mapping = preprocess_dataframe(df, uci_datasets_info[name]['target_variable'][0], uci_datasets_info[name]['categorical_variables'], uci_datasets_info[name]['id_variables'])
        # print(df.shape)

        # print(np.isinf(df).sum())
        
        tensor_data = torch.tensor(df.to_numpy(), dtype=torch.float32, device='cuda')

        if df['x0'].unique().shape[0] <= 2:
            col_width = 0.5
        else:
            IQR = df['x0'].quantile(0.75) - df['x0'].quantile(0.25)
            col_width = 2 * (IQR / math.cbrt(tensor_data.shape[0]))

        num_bins = math.ceil((df['x0'].max() - df['x0'].min()) / col_width)

        print(name, num_bins)
        
        
        # col_width = (tensor_data[0].max(dim=0).values - tensor_data.min(dim=0).values) / num_bins
        # col_width = (tensor_data[0].max() - tensor_data[0].min()) / num_bins
        pos = tensor_data[:, 1:].contiguous()
        tar = tensor_data[:, 0].unsqueeze(1).contiguous()

        hist_idx = torch.clamp((tar // col_width).long().squeeze(-1), max=num_bins - 1)
        hists = torch.zeros((df.shape[0], num_bins), device='cuda')
        hists[torch.arange(df.shape[0]), hist_idx] = 1.0

        for p in pocket_sizes:
            # if os.path.isfile(f'./perceptive_graphs/{name}_{p}.pkl'):
            #     # print(f"{name}_{p}.pkl exists. Skipping...")
            #     if name != 'sepsis_survival_primary_cohort':
            #         continue
            
            cl, c, duration = KMeans(pos, K=pos.shape[0] // p, Niter=10000)
            
            pockets = []
            for ci in cl.unique():
                pockets.append(torch.nonzero(cl == ci).squeeze(-1))

            cl_index = cl.unsqueeze(1).expand(-1, hists.shape[1])
            cl_hists = torch.zeros(pos.shape[0], hists.shape[1], device='cuda')
            cl_hists.scatter_add_(0, cl_index, hists)

            if len(pockets) < pos.shape[0]:
                cl_hists = cl_hists[cl.unique()]
                c = c[cl.unique()]
                
            with open(f'./perceptive_graphs/{name}_{p}.pkl', 'wb') as f:
                pickle.dump({'col_width': col_width, 'num_bins': num_bins, 'pockets': pockets, 'midpoints': c, 'pocket_hists': cl_hists, 'time': duration}, f)
            
    except Exception as e:
        print(e)
        print(df.dtypes)

insurance 30
K-means for the Manhattan metric with 1,338 points in dimension 4, K = 1,338:
Timing for 10000 iterations: 3.15686s = 10000 x 0.00032s

K-means for the Manhattan metric with 1,338 points in dimension 4, K = 669:
Timing for 10000 iterations: 2.83985s = 10000 x 0.00028s

K-means for the Manhattan metric with 1,338 points in dimension 4, K = 334:
Timing for 10000 iterations: 2.69138s = 10000 x 0.00027s

Converged at iteration 12
K-means for the Manhattan metric with 1,338 points in dimension 4, K = 167:
Timing for 10000 iterations: 0.00407s = 10000 x 0.00000s

K-means for the Manhattan metric with 1,338 points in dimension 4, K = 83:
Timing for 10000 iterations: 2.56841s = 10000 x 0.00026s

Converged at iteration 19
K-means for the Manhattan metric with 1,338 points in dimension 4, K = 41:
Timing for 10000 iterations: 0.00566s = 10000 x 0.00000s

Converged at iteration 20
K-means for the Manhattan metric with 1,338 points in dimension 4, K = 20:
Timing for 10000 iterations: 0